In [1]:
# ── CELL 1: Imports and GEE Initialisation ──────────────────────────────────
import ee
import geemap
import os

ee.Initialize(project='ee-festac')

# Load Amuwo Odofin boundary — note the leading space in ' Amuwo Odofin'
# This is a GAUL dataset quirk discovered during study area definition
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.eq('ADM2_NAME', 'Amuwo Odofin'))

# Always verify boundary loaded before extracting geometry
# A silent empty collection is harder to debug than an explicit check
feature_count = amuwo_odofin.size().getInfo()
print(f"Amuwo Odofin feature count: {feature_count}")

if feature_count == 0:
    raise ValueError("Boundary returned 0 features — check ADM2_NAME filter spelling")

# Extract geometry only after confirming the collection is not empty
amuwo_geom = amuwo_odofin.geometry()

print("✓ GEE initialised")
print("✓ Amuwo Odofin boundary loaded and verified")
print("✓ Geometry extracted — ready for DEM acquisition")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Amuwo Odofin feature count: 1
✓ GEE initialised
✓ Amuwo Odofin boundary loaded and verified
✓ Geometry extracted — ready for DEM acquisition


In [2]:
# ── CELL 2: Load SRTM 30m DEM and Clip to Study Area ────────────────────────
# Source: NASA/USGS Shuttle Radar Topography Mission
# Resolution: 30 metres | Coverage: Global
# GEE ID: USGS/SRTMGL1_003

# Load global DEM
srtm = ee.Image("USGS/SRTMGL1_003")

# Clip to Amuwo Odofin boundary
dem = srtm.clip(amuwo_geom)

# Get basic statistics
dem_stats = dem.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9
).getInfo()

print("✓ SRTM DEM loaded and clipped to Amuwo Odofin")
print()
print("Elevation Statistics (metres):")
print(f"  Minimum : {dem_stats['elevation_min']:.2f} m")
print(f"  Maximum : {dem_stats['elevation_max']:.2f} m")
print(f"  Mean    : {dem_stats['elevation_mean']:.2f} m")

✓ SRTM DEM loaded and clipped to Amuwo Odofin

Elevation Statistics (metres):
  Minimum : -38.00 m
  Maximum : 70.00 m
  Mean    : 5.77 m


In [3]:
# ── CELL 3: Derive All Topographic Conditioning Factors ─────────────────────
# These become features 1-6 in the ML feature matrix

# ── 1. Elevation (raw DEM) ───────────────────────────────────────────────────
elevation = dem.rename('elevation')

# ── 2. Slope (degrees) ──────────────────────────────────────────────────────
slope = ee.Terrain.slope(dem).rename('slope')

# ── 3. Aspect (degrees from north) ──────────────────────────────────────────
aspect = ee.Terrain.aspect(dem).rename('aspect')

# ── 4. Flow Accumulation ─────────────────────────────────────────────────────
# Using HydroSHEDS hydrologically conditioned flow accumulation
# Better than DEM-derived for flat coastal terrain like Amuwo Odofin
flow_acc = ee.Image("WWF/HydroSHEDS/15ACC") \
             .clip(amuwo_geom) \
             .rename('flow_accumulation')

# ── 5. Topographic Wetness Index (TWI) ───────────────────────────────────────
# TWI = ln(flow_accumulation / tan(slope_in_radians))
# High TWI = more likely to accumulate water = higher flood risk

# Convert slope to radians for TWI calculation
slope_rad = slope.multiply(ee.Number(3.14159265).divide(180))

# Avoid division by zero on flat areas — set minimum slope to 0.001 radians
slope_rad_safe = slope_rad.where(slope_rad.lt(0.001), 0.001)

# Compute TWI
twi = flow_acc.divide(slope_rad_safe).log().rename('twi')

# ── 6. Plan Curvature (manually computed via Laplacian kernel) ───────────────
# ee.Terrain.products() only returns elevation/slope/aspect/hillshade
# so I compute curvature myself using a Laplacian convolution
# Laplacian detects rate of slope change across neighbouring pixels:
# negative values = concave terrain (water concentrates = higher flood risk)
# positive values = convex terrain (water disperses = lower flood risk)

laplacian_kernel = ee.Kernel.laplacian8(normalize=False)
curvature = elevation.convolve(laplacian_kernel) \
                     .rename('curvature') \
                     .clip(amuwo_geom)

# ── Verify all layers ────────────────────────────────────────────────────────
print("✓ Topographic features derived:")
print("  1. Elevation")
print("  2. Slope")
print("  3. Aspect")
print("  4. Flow Accumulation (HydroSHEDS)")
print("  5. Topographic Wetness Index (TWI)")
print("  6. Curvature")

✓ Topographic features derived:
  1. Elevation
  2. Slope
  3. Aspect
  4. Flow Accumulation (HydroSHEDS)
  5. Topographic Wetness Index (TWI)
  6. Curvature


In [4]:
# ── CELL 4: Visualise Topographic Layers ─────────────────────────────────────

# Get centroid for map centering
amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=12)

# Study area boundary
amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# Elevation — blue (low) to red (high)
Map.addLayer(elevation, {
    'min': 0, 'max': 50,
    'palette': ['0000FF', '00FFFF', '00FF00', 'FFFF00', 'FF0000']
}, 'Elevation (m)')

# Slope — white (flat) to dark brown (steep)
Map.addLayer(slope, {
    'min': 0, 'max': 15,
    'palette': ['FFFFFF', 'F5DEB3', 'A0522D', '4B0000']
}, 'Slope (degrees)')

# TWI — yellow (dry) to dark blue (wet/flood-prone)
Map.addLayer(twi, {
    'min': 0, 'max': 15,
    'palette': ['FFFF00', '00FF00', '0000FF', '000080']
}, 'Topographic Wetness Index (TWI)')

# Flow Accumulation — log scale for visibility
Map.addLayer(flow_acc.log(), {
    'min': 0, 'max': 10,
    'palette': ['FFFFFF', 'ADD8E6', '0000FF', '000033']
}, 'Flow Accumulation (log scale)')

Map.addLayerControl()
Map

Map(center=[6.455061708555176, 3.2650782915242123], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
# ── CELL 5: Stack Features and Export to Google Drive ───────────────────────
# All topographic conditioning factors are cast to Float32 before stacking
# to ensure type consistency across bands from different source datasets.
# SRTM-derived bands are Int16 by default; TWI and curvature are Float32.
# Standardising to Float32 preserves decimal precision across all features,
# which is required for the downstream ML feature matrix.

topo_stack = elevation.toFloat() \
    .addBands(slope.toFloat()) \
    .addBands(aspect.toFloat()) \
    .addBands(flow_acc.toFloat()) \
    .addBands(twi.toFloat()) \
    .addBands(curvature.toFloat())

# Confirm all band names before exporting
print("Stacked band names:", topo_stack.bandNames().getInfo())

# Export the multi-band stack to Google Drive as a GeoTIFF
# Creates a GeoTIFF in your Google Drive under GeoAID_Project/
# Scale=30 matches the native SRTM resolution
# EPSG:4326 is WGS84 geographic coordinate system
# This runs as a background GEE task — check progress at code.earthengine.google.com

export_task = ee.batch.Export.image.toDrive(
    image=topo_stack,
    description='amuwo_odofin_topographic_features',
    folder='GeoAID_Project',
    fileNamePrefix='topo_features_amuwo_odofin',
    region=amuwo_geom,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

export_task.start()
print()
print("✓ Export task submitted to Google Drive")
print("  Folder : GeoAID_Project/")
print("  File   : topo_features_amuwo_odofin.tif")
print("  Scale  : 30 metres | Types: Float32")
print()
print("Monitor progress at: https://code.earthengine.google.com/tasks")

Stacked band names: ['elevation', 'slope', 'aspect', 'flow_accumulation', 'twi', 'curvature']

✓ Export task submitted to Google Drive
  Folder : GeoAID_Project/
  File   : topo_features_amuwo_odofin.tif
  Scale  : 30 metres | Types: Float32

Monitor progress at: https://code.earthengine.google.com/tasks


In [6]:
# ── CELL 6: Session Summary ──────────────────────────────────────────────────
print("=" * 55)
print("  NOTEBOOK 02 — DEM ACQUISITION: COMPLETE")
print("=" * 55)
print()
print("  Datasets acquired:")
print("  ├── SRTM 30m DEM (NASA/USGS) ✓")
print("  └── HydroSHEDS Flow Accumulation (WWF) ✓")
print()
print("  Features derived:")
print("  ├── Elevation ✓")
print("  ├── Slope ✓")
print("  ├── Aspect ✓")
print("  ├── Flow Accumulation ✓")
print("  ├── Topographic Wetness Index (TWI) ✓")
print("  └── Curvature ✓")
print()
print("  Export: GeoAID_Project/ on Google Drive ✓")
print()
print("  Next: 03_rainfall_lulc_acquisition.ipynb")
print("        → CHIRPS rainfall + ESA WorldCover LULC")
print("        → Sentinel-2 NDVI")
print("=" * 55)

  NOTEBOOK 02 — DEM ACQUISITION: COMPLETE

  Datasets acquired:
  ├── SRTM 30m DEM (NASA/USGS) ✓
  └── HydroSHEDS Flow Accumulation (WWF) ✓

  Features derived:
  ├── Elevation ✓
  ├── Slope ✓
  ├── Aspect ✓
  ├── Flow Accumulation ✓
  ├── Topographic Wetness Index (TWI) ✓
  └── Curvature ✓

  Export: GeoAID_Project/ on Google Drive ✓

  Next: 03_rainfall_lulc_acquisition.ipynb
        → CHIRPS rainfall + ESA WorldCover LULC
        → Sentinel-2 NDVI
